In [1]:
from datasets import load_dataset
dataset_path = "dataset/dataset_curation_agent/unit_tests_output.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

dataset = dataset.shuffle(seed=42)

total_len = len(dataset)
train_end = int(0.7 * total_len)
eval_end = int(0.9 * total_len)

# Slice the dataset
train_dataset = dataset.select(range(0, train_end))
eval_dataset  = dataset.select(range(train_end, eval_end))
test_dataset  = dataset.select(range(eval_end, total_len))

# Print lengths to verify
print("Total:", total_len)
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Test:", len(test_dataset))

print("Train:","example: ",train_dataset[0],"\n")
print("Eval:", len(eval_dataset),"\n example: ",eval_dataset[0],"\n")
print("Test:", len(test_dataset),"\n example: ",test_dataset[0],"\n")

Total: 10
Train: 7
Eval: 2
Test: 1
Train: example:  {'task': 'Fix the issue in the following Python code.', 'buggy_code': 'def twill_go_with_relative_paths(uri, *args, **kwargs):\n    if not uri.startswith("http"):\n        base = get_live_server_path()\n        if uri.startswith("/"):\n            base = base.rstrip("/")\n        uri = "%s%s" * (base, uri)\n    response = original_go(uri, *args, **kwargs)\n    if browser.result.get_http_code() == 500:\n        raise extract_django_traceback(twill=browser)\n    else:\n        return response', 'correct_code': 'def twill_go_with_relative_paths(uri, *args, **kwargs):\n    if not uri.startswith("http"):\n        base = get_live_server_path()\n        if uri.startswith("/"):\n            base = base.rstrip("/")\n        uri = "%s%s" % (base, uri)\n    response = original_go(uri, *args, **kwargs)\n    if browser.result.get_http_code() == 500:\n        raise extract_django_traceback(twill=browser)\n    else:\n        return response', 'unit_

In [2]:
EVAL_REFERENCES = [ex["correct_code"] for ex in eval_dataset]
TEST_REFERENCES = [ex["correct_code"] for ex in test_dataset]
print("eval_references:", EVAL_REFERENCES[0],"\n")
print("test_references:", TEST_REFERENCES[0],"\n")

eval_references: def validate_file(self,filename):
    import os.path
    base,ext = os.path.splitext(filename)
    if ext != '.py': return         # No idea what the file is. Return OK

    try:
        f = open(filename)
        lines = f.readlines()
        f.close()
    except IOError:
        return                      # Couldn't find the file.  Don't worry about it

    fre = re.compile(r'\s*def\s+(t_[a-zA-Z_0-9]*)\(')
    sre = re.compile(r'\s*(t_[a-zA-Z_0-9]*)\s*=')

    counthash = { }
    linen = 1
    for l in lines:
        m = fre.match(l)
        if not m:
            m = sre.match(l)
        if m:
            name = m.group(1)
            prev = counthash.get(name)
            if not prev:
                counthash[name] = linen
            else:
                self.log.error("%s:%d: Rule %s redefined. Previously defined on line %d",filename,linen,name,prev)
                self.error = 1
        linen += 1 

test_references: @expose('/',methods=('GET','POST',))
def in

In [3]:
#Just for my train split
def formatting_prompts_func(examples):
    output_text = []
    for i in range(len(examples["task"])):
        instruction = examples["task"][i]
        input_code = examples["buggy_code"][i]
        response = examples["correct_code"][i]

        if input_code.strip():  # if not empty
            text = f'''Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

        ### Instruction:
        {instruction}

        ### Input:
        {input_code}

        ### Response:
        {response}
        '''

        output_text.append(text)

    return output_text


In [4]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


In [5]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    max_seq_length = 4096,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.2: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 2. Max memory: 11.994 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2025.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [7]:
# import neptune
# import neptune.integrations.optuna as optuna_utils
# run = neptune.init_run(
#     project="casvi/CodeMedic",
#     api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIzMTMzYjhhOC1jYzA1LTQ0YjAtOTJjNi1iY2EzM2VhMDY0OTcifQ=="
# )


In [8]:
import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
import evaluate
from codebleu import compute_codebleu

# Cargar métricas estándar
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
acc = evaluate.load("accuracy")
code_eval = evaluate.load("code_eval")


# Preprocesar logits
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


# Función principal de métricas para el Trainer
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = labels[:, 1:]
    preds = preds[:, :-1]

    # Handle padding/masks
    mask = labels == -100
    labels[mask] = tokenizer.pad_token_id
    preds[mask] = tokenizer.pad_token_id

    # Decode tokens
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Extract completions from "##Fixed Code:"
    decoded_completions = []
    for pred in decoded_preds:
        parts = pred.split("##Fixed Code:")
        completion = parts[-1].strip() if len(parts) > 1 else pred.strip()
        decoded_completions.append(completion)

    # Standard metrics
    bleu_score = bleu.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    rouge_score = rouge.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])

    # CodeBLEU
    refs = [[ref] for ref in EVAL_REFERENCES]
    codebleu_scores = compute_codebleu(decoded_completions, refs, lang="python")

    #Pass@k
    #Convert to shape: candidates = [[completion1], [completion2], ...]
    predictions = [[pred] for pred in decoded_completions]
    pass_at_k_result, _ = code_eval.compute(
        references=EVAL_REFERENCES,
        predictions=predictions,
        k=[1, 2, 4, 8]
    )

    return {
        "codebleu": codebleu_scores["codebleu"],
        **bleu_score,
        **rouge_score,
        **accuracy,
        **{f"pass@{k}": v for k, v in pass_at_k_result.items()},
    }

In [9]:
from trl import SFTTrainer, SFTConfig
import time
start=time.time()

# SFT Config
config = SFTConfig(
    dataset_num_proc = 1,
    #dataset_text_field="prompt",#Depends on the colum of your data set
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=5,
    report_to="none",
    logging_steps=100,
    max_steps=2000,
    eval_accumulation_steps=100,
)
trainer = SFTTrainer(
    model=model,  # base or PEFT model
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_prompts_func,
    args=config,
    warmup_steps = 5,
    weight_decay = 0.01,
    compute_metrics = compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)
metrics = trainer.evaluate()
print("Metrics:",metrics)
trainer.train()


end = time.time()
length = end - start

hours = int(length // 3600)
minutes = int((length % 3600) // 60)
seconds = int(length % 60)

print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")


Unsloth: Tokenizing ["text"]:   0%|          | 0/2 [00:00<?, ? examples/s]

NotImplementedError: This metric is currently not supported on Windows.

In [10]:
metrics = trainer.evaluate()
print("Metrics:",metrics)


In [11]:
# from trl import SFTTrainer, SFTConfig
# import time
# def objective(trial):
#     start=time.time()
#     # Suggest hyperparameters
#     learning_rate = trial.suggest_float("learning_rate", 1e-5,5e-4, log=True)
#     num_epochs = trial.suggest_int("num_train_epochs", 5, 10)
#     max_steps = trial.suggest_int("max_steps", 500,2000)
#     batch_size=2
#
#     # SFT Config
#     config = SFTConfig(
#         dataset_num_proc = 1,
#         #output_dir="./outputs",
#         dataset_text_field="text",#Depends on the colum of your data set
#         learning_rate=learning_rate,
#         per_device_train_batch_size=batch_size,
#         gradient_accumulation_steps=batch_size,
#         num_train_epochs=num_epochs,
#         report_to="none",
#         logging_steps=100,
#         max_steps=max_steps,
#         eval_accumulation_steps=100
#     )
#
#     trainer = SFTTrainer(
#         model=model,  # base or PEFT model
#         tokenizer=tokenizer,
#         train_dataset=dataset,
#         eval_dataset=eval_dataset,
#         args=config,
#         warmup_steps = 5,
#         weight_decay = 0.01,
#         compute_metrics = compute_metrics,
#         preprocess_logits_for_metrics=preprocess_logits_for_metrics,
#     )
#     trainer.train()
#     metrics = trainer.evaluate()
#     print("Metrics:",metrics)
#
#     # Log trial info to Neptune
#     run[f"trial/{trial.number}/metrics"] = metrics
#     run[f"trial/{trial.number}/params"] = {
#         "learning_rate": learning_rate,
#         "num_epochs": num_epochs,
#         "max_steps": max_steps,
#     }
#
#
#     end = time.time()
#     length = end - start
#
#     hours = int(length // 3600)
#     minutes = int((length % 3600) // 60)
#     seconds = int(length % 60)
#
#     print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")
#
#     return metrics["eval_loss"]  # Or any other metric


In [12]:
# import optuna
# start=time.time()
# neptune_callback = optuna_utils.NeptuneCallback(run)
#
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=6, callbacks=[neptune_callback], show_progress_bar=True)
# end = time.time()
# length = end - start
#
# hours = int(length // 3600)
# minutes = int((length % 3600) // 60)
# seconds = int(length % 60)
# print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

[I 2025-05-17 17:52:31,709] A new study created in memory with name: no-name-de451807-72fd-42eb-a111-c2af1693acf3


  0%|          | 0/6 [00:00<?, ?it/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,000 | Num Epochs = 1 | Total steps = 1,640
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 69,730,304/7,000,000,000 (1.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
100,1.254600
200,1.068900
300,1.034000
400,1.033200
500,1.015400
600,1.005500
700,0.976300
800,0.966800
900,1.003600
1000,0.956300


...

Step,Training Loss
100,0.430800
200,0.399900
300,0.398600
400,0.411000
500,0.427300
600,0.436200
700,0.433900
800,0.455000


[DEBUG] BLEU: 1.0, Weighted BLEU: 0.2215311839180054, CodeBLEU: 0.6107655919590027
Metrics: {'eval_loss': 1.196340560913086, 'eval_codebleu': 0.6107655919590027, 'eval_bleu': 0.8576592353751195, 'eval_precisions': [0.9313146658380232, 0.8765484656249625, 0.8343199339207048, 0.7980376766091052], 'eval_brevity_penalty': 0.9988670120521486, 'eval_length_ratio': 0.9988676533984768, 'eval_translation_length': 173778, 'eval_reference_length': 173975, 'eval_rouge1': 0.8660387187722581, 'eval_rouge2': 0.7516715490347392, 'eval_rougeL': 0.8645750528648277, 'eval_rougeLsum': 0.8646137152482134, 'eval_accuracy': 0.7975672173194717, 'eval_runtime': 118.0052, 'eval_samples_per_second': 59.319, 'eval_steps_per_second': 7.415}
It took 0 hours, 14 minutes, and 13 seconds to train the model!
[I 2025-05-17 19:28:23,513] Trial 4 finished with value: 1.196340560913086 and parameters: {'learning_rate': 2.1039966762818298e-05, 'num_train_epochs': 7, 'max_steps': 858}. Best is trial 0 with value: 1.001642942

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,000 | Num Epochs = 1 | Total steps = 1,305
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 69,730,304/7,000,000,000 (1.00% trained)


Step,Training Loss
100,0.272200
200,0.302000
300,0.322000
400,0.356700
500,0.389200
600,0.414500
700,0.422600
800,0.453500
900,0.506700
1000,0.535600


[DEBUG] BLEU: 1.0, Weighted BLEU: 0.2212344892317695, CodeBLEU: 0.6106172446158847
Metrics: {'eval_loss': 1.1274174451828003, 'eval_codebleu': 0.6106172446158847, 'eval_bleu': 0.8578303293646549, 'eval_precisions': [0.9322659814720929, 0.8776294304101431, 0.8355784171219488, 0.7993935106954323], 'eval_brevity_penalty': 0.9977039351087621, 'eval_length_ratio': 0.9977065670354937, 'eval_translation_length': 173576, 'eval_reference_length': 173975, 'eval_rouge1': 0.8669250639035307, 'eval_rouge2': 0.7543261518462463, 'eval_rougeL': 0.8656601960549244, 'eval_rougeLsum': 0.8657405321871265, 'eval_accuracy': 0.7986879071406427, 'eval_runtime': 119.9707, 'eval_samples_per_second': 58.348, 'eval_steps_per_second': 7.293}
It took 0 hours, 21 minutes, and 0 seconds to train the model!
[I 2025-05-17 19:49:24,779] Trial 5 finished with value: 1.1274174451828003 and parameters: {'learning_rate': 6.0748157401137604e-05, 'num_train_epochs': 9, 'max_steps': 1305}. Best is trial 0 with value: 1.0016429

In [13]:
# # Get the best parameters
# best_trial = study.best_trial
#
# best_params = best_trial.params
# print("best_params: ",best_params)
#
# best_value = best_trial.value
# print("Eval loss:", best_value)
# run.stop()

best_params:  {'learning_rate': 3.661300350919904e-05, 'num_train_epochs': 6, 'max_steps': 1640}
Eval loss: 1.0016429424285889
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 60 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 60 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/casvi/CodeMedic/e/COD-82/metadata
